# DDRI 군집별 모델링 템플릿(Cluster Modeling Template)

이 노트북은 `works/05_prediction_long/05_ddri_team_cluster_modeling_protocol.md` 기준으로, 대표 대여소 15개 중 특정 `station_group` 1개를 맡아 공통 실험을 수행하기 위한 템플릿이다.

사용 방법:

1. 이 파일을 복사한다.
2. 파일명을 담당 군집에 맞게 바꾼다.
3. `TARGET_STATION_GROUP` 값을 본인 군집명으로 바꾼다.
4. 아래 순서를 그대로 따라 실험한다.


## 1. 실험 목적(Experiment Goal)

- 담당 군집: `여기에 군집명 기입`
- 목표: 대표 대여소 3개를 대상으로 `2023 train / 2024 validation / 2025 test` 정책에 따라 공통 모델 실험을 수행한다.
- 필수 모델: `LinearRegression`, `LightGBM_RMSE`
- 필수 평가 지표: `RMSE`, `MAE`, `R²`


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

try:
    from lightgbm import LGBMRegressor
except ImportError:
    LGBMRegressor = None

BASE_DIR = Path('/Users/cheng80/Desktop/ddri_work/works/05_prediction_long')
TRAIN_PATH = BASE_DIR / 'data' / 'ddri_prediction_long_train_2023_2024.csv'
TEST_PATH = BASE_DIR / 'data' / 'ddri_prediction_long_test_2025.csv'
OUTPUT_DIR = BASE_DIR / 'output' / 'team_cluster_template_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_STATION_GROUP = '업무/상업 혼합형'  # 여기를 담당 군집명으로 수정
DISPLAY_NAME = TARGET_STATION_GROUP

RANDOM_STATE = 42


## 2. 입력 데이터 로드(Input Data Load)

- 공통 입력 정본은 `works/05_prediction_long/data/` 아래 CSV 2개다.
- 개인이 별도 가공한 CSV를 정본처럼 사용하지 않는다.


In [ ]:
train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)

print('train shape:', train_raw.shape)
print('test shape:', test_raw.shape)
print('station groups:', sorted(train_raw['station_group'].dropna().unique().tolist()))


## 3. 담당 군집 필터링(Filter Assigned Cluster)

- 반드시 `station_group` 1개만 남긴다.
- 결과적으로 대표 대여소 3개만 남아야 한다.


In [ ]:
train_group = train_raw.loc[train_raw['station_group'] == TARGET_STATION_GROUP].copy()
test_group = test_raw.loc[test_raw['station_group'] == TARGET_STATION_GROUP].copy()

print('train_group shape:', train_group.shape)
print('test_group shape:', test_group.shape)
print(train_group[['station_id', 'station_name']].drop_duplicates().sort_values('station_id').to_string(index=False))


## 4. 전처리 및 파생 피처 생성(Preprocessing And Feature Engineering)

공통 프로토콜:

- `date`는 날짜형으로 변환한다.
- `station_id`, `date`, `hour` 기준으로 정렬한다.
- `lag_1h`, `lag_24h`, `lag_168h`를 만든다.
- `rolling_mean_24h`, `rolling_std_24h`, `rolling_mean_168h`, `rolling_std_168h`를 만든다.
- rolling은 반드시 `station_id`별로 계산하고, `shift(1)` 후 rolling을 적용한다.
- `lag_168h`는 `1주 전 동일 요일·동일 시각 대여량`이다.


In [ ]:
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy()
    result['date'] = pd.to_datetime(result['date'])
    result = result.sort_values(['station_id', 'date', 'hour']).reset_index(drop=True)

    group = result.groupby('station_id', group_keys=False)['rental_count']
    result['lag_1h'] = group.shift(1)
    result['lag_24h'] = group.shift(24)
    result['lag_168h'] = group.shift(168)

    shifted = group.shift(1)
    result['rolling_mean_24h'] = shifted.rolling(24).mean().reset_index(level=0, drop=True)
    result['rolling_std_24h'] = shifted.rolling(24).std().reset_index(level=0, drop=True)
    result['rolling_mean_168h'] = shifted.rolling(168).mean().reset_index(level=0, drop=True)
    result['rolling_std_168h'] = shifted.rolling(168).std().reset_index(level=0, drop=True)
    return result

train_feat = build_features(train_group)
test_feat = build_features(test_group)

train_feat[['station_id', 'date', 'hour', 'rental_count', 'lag_1h', 'lag_24h', 'lag_168h']].head()


## 5. 시간 분할 정책(Time-Based Split Policy)

- 학습(`Train`): `2023`
- 검증(`Validation`): `2024`
- 테스트(`Test`): `2025`

랜덤 분할은 금지한다.


In [ ]:
train_2023 = train_feat.loc[train_feat['date'].dt.year == 2023].copy()
valid_2024 = train_feat.loc[train_feat['date'].dt.year == 2024].copy()
test_2025 = test_feat.copy()

print('train_2023:', train_2023.shape)
print('valid_2024:', valid_2024.shape)
print('test_2025:', test_2025.shape)


## 6. 입력 피처 목록(Input Feature List)

아래 목록은 공통 기본 피처다. 개인 실험에서 피처를 추가하거나 제거했다면 반드시 이유를 마크다운에 남긴다.


In [ ]:
target_col = 'rental_count'

categorical_features = [
    'station_id',
    'cluster',
    'mapped_dong_code',
]

numeric_features = [
    'hour', 'weekday', 'month', 'holiday',
    'temperature', 'humidity', 'precipitation', 'wind_speed',
    'lag_1h', 'lag_24h', 'lag_168h',
    'rolling_mean_24h', 'rolling_std_24h',
    'rolling_mean_168h', 'rolling_std_168h',
]

feature_cols = categorical_features + numeric_features
feature_cols


## 7. 평가 함수(Evaluation Function)

모든 팀원은 아래 3개 점수를 동일하게 기록한다.

- `RMSE`
- `MAE`
- `R²`


In [ ]:
def evaluate_regression(y_true, y_pred):
    return {
        'rmse': float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'mae': float(mean_absolute_error(y_true, y_pred)),
        'r2': float(r2_score(y_true, y_pred)),
    }

def build_linear_pipeline():
    preprocessor = ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
            ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), numeric_features),
        ]
    )
    return Pipeline([
        ('preprocessor', preprocessor),
        ('model', LinearRegression()),
    ])

def build_lgbm_model():
    if LGBMRegressor is None:
        raise ImportError('lightgbm 패키지가 설치되어 있어야 합니다.')
    return LGBMRegressor(
        objective='regression',
        n_estimators=400,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_STATE,
    )


## 8. 모델 학습(Model Training)

필수 순서:

1. `LinearRegression`
2. `LightGBM_RMSE`

validation 결과를 기준으로 우세 모델을 선택한다.


In [ ]:
results = []

X_train = train_2023[feature_cols]
y_train = train_2023[target_col]
X_valid = valid_2024[feature_cols]
y_valid = valid_2024[target_col]

linear_model = build_linear_pipeline()
linear_model.fit(X_train, y_train)
linear_pred = linear_model.predict(X_valid)
linear_metrics = evaluate_regression(y_valid, linear_pred)
results.append({
    'station_group': TARGET_STATION_GROUP,
    'model': 'LinearRegression',
    'split': 'validation_2024',
    **linear_metrics,
})

if LGBMRegressor is not None:
    lgbm_model = build_lgbm_model()
    lgbm_model.fit(X_train, y_train)
    lgbm_pred = lgbm_model.predict(X_valid)
    lgbm_metrics = evaluate_regression(y_valid, lgbm_pred)
    results.append({
        'station_group': TARGET_STATION_GROUP,
        'model': 'LightGBM_RMSE',
        'split': 'validation_2024',
        **lgbm_metrics,
    })

validation_results = pd.DataFrame(results)
validation_results


## 9. 우세 모델 선택 및 재학습(Refit Best Model)

- validation 결과를 기준으로 우세 모델을 선택한다.
- 선택 모델을 `2023 + 2024`로 재학습한다.
- 그 다음 `2025` 테스트셋 점수를 산출한다.


In [ ]:
best_model_name = validation_results.sort_values('rmse').iloc[0]['model']
print('best_model_name:', best_model_name)

refit_train = train_feat.copy()
X_refit = refit_train[feature_cols]
y_refit = refit_train[target_col]
X_test = test_2025[feature_cols]
y_test = test_2025[target_col]

if best_model_name == 'LinearRegression':
    best_model = build_linear_pipeline()
else:
    best_model = build_lgbm_model()

best_model.fit(X_refit, y_refit)
test_pred = best_model.predict(X_test)
test_metrics = evaluate_regression(y_test, test_pred)

test_results = pd.DataFrame([{ 
    'station_group': TARGET_STATION_GROUP,
    'model': best_model_name,
    'split': 'test_2025_refit',
    **test_metrics,
}])
test_results


## 10. 결과 저장(Result Save)

- 결과 CSV는 담당 군집명이 보이게 저장한다.
- 최소 2개 split(`validation_2024`, `test_2025_refit`)이 모두 있어야 한다.


In [ ]:
final_metrics = pd.concat([validation_results, test_results], ignore_index=True)
safe_group_name = TARGET_STATION_GROUP.replace('/', '_').replace(' ', '_')
metrics_path = OUTPUT_DIR / f'ddri_{safe_group_name}_model_metrics.csv'
final_metrics.to_csv(metrics_path, index=False)
print('saved:', metrics_path)
final_metrics


## 11. 결과 해석(Result Interpretation)

아래 항목을 팀원별로 반드시 직접 작성한다.

- validation 기준 우세 모델은 무엇인가
- test 점수는 어느 정도인가
- 이 군집에서 예측이 어려운 시간대가 보이는가
- 날씨나 1주 전 동일 시각(`lag_168h`) 영향이 큰 것처럼 보이는가
- 다음 실험에서 추가할 피처 또는 제거할 피처는 무엇인가


## 12. 제출 전 체크리스트(Submission Checklist)

- [ ] 담당 군집 3개 대여소만 사용했는가
- [ ] `2023 train / 2024 validation / 2025 test`를 지켰는가
- [ ] `LinearRegression`과 `LightGBM_RMSE`를 모두 돌렸는가
- [ ] `RMSE`, `MAE`, `R²`를 validation/test 모두 기록했는가
- [ ] lag/rolling을 `station_id`별로 만들었는가
- [ ] 테스트셋으로 모델 선택을 하지 않았는가
- [ ] 노트북 마크다운에 전처리와 해석이 남아 있는가
- [ ] 차트와 표기가 `한글(영문)` 형식인가
